In [1]:
%load_ext autoreload
%autoreload 2

# System libraries.
import logging

# Third party libraries.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# System libraries.
import asyncio
import os

# Third party libraries.
from dataclasses import dataclass

import nest_asyncio
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai import ModelRetry

# Local utilities.
import pydanticai_API_utils as utils

# Notebook-specific imports are ready for tutorial examples.

In [3]:
# Configure notebook logging.
import logging

# Local utility.
import pydanticai_API_utils as utils

_LOG = logging.getLogger(__name__)
utils.init_logger(_LOG)
_LOG
# Notebook logging is configured for the tutorial cells.

INFO  > cmd='/opt/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-9b586c65-e2dd-4a36-8b3c-1ffb500bfed5.json'


# Summary

- This notebook introduces `PydanticAI` APIs for building LLM workflows, including structured outputs, tools, dependencies, validators, streaming, provider configuration, run metadata, and usage limits

# PydanticAI API Introduction

- `PydanticAI` is a lightweight framework for building LLM-powered applications with structured outputs
- `PydanticAI` uses `Pydantic` models to define response schemas
- Traditional LLM APIs often return unstructured text
- `PydanticAI` keeps responses aligned with a predefined schema

## Why PydanticAI Exists

- Key problem: LLMs typically return unstructured text
- Example prompt:
  - "Extract product information from this description"
- Example LLM output:
  - "The product is an iPhone 15 priced at $999."
- Problem with the example LLM output:
  - The example LLM output is difficult to use programmatically
- Desired structured output:

  ```json
  {
    "product_name": "iPhone 15",
    "price": 999
  }
  ```

- `PydanticAI` solves this problem with:
  - Schema definitions with `Pydantic` models
  - Structured output enforcement
  - Automatic retries after validation failures
  - A simple agent abstraction for LLM interaction

## Mental Model

- `PydanticAI` flow:
  ```mermaid
  flowchart TD
      A[User Prompt] --> B[PydanticAI Agent]
      B --> C[LLM]
      C --> D[Raw Response]
      D --> E[Pydantic Validation]
      E --> F[Structured Output]
  ```

In [4]:
# Load environment variables from a local dotenv file if one exists.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)
_LOG.info("dotenv path: %s", env_path or "<not found>")
env_path or "<not found>"
# Environment variables are available to the model configuration cells.

dotenv path: /git_root/tutorials/tutorial_pydanticAI/.env


'/git_root/tutorials/tutorial_pydanticAI/.env'

In [5]:
# Read the model identifier from the environment.
MODEL_ID = os.getenv("PYDANTIC_AI_MODEL")
utils.log_environment(env_path, MODEL_ID)
{"model_id": MODEL_ID}
# The tutorial examples will use the configured model identifier.

dotenv path: /git_root/tutorials/tutorial_pydanticAI/.env
PYDANTIC_AI_MODEL: openai:gpt-4.1-mini
OPENAI_API_KEY: sk-...8A


{'model_id': 'openai:gpt-4.1-mini'}

# Core Concepts

- `PydanticAI` revolves around a few important abstractions

## Agent

- `Agent` is the main interface for interacting with the model
- `Agent` manages:
  - LLM calls
  - Structured outputs
  - Retries
  - Tool usage

## output_type

- `output_type` defines the expected structured output
- `output_type` must be a `Pydantic` model

## Tools

- Tools are functions that the agent can call during reasoning
- Tools let agents interact with external systems such as APIs or databases



# Minimal Example

- The quickest way to understand `PydanticAI` is a small example
- This section defines a schema with `Pydantic` and asks the agent to produce that structured output

In [6]:
# Define the output schema for the minimal example.
class City(BaseModel):
    name: str
    country: str
    population: int


City
# The schema defines the exact output shape expected from the model.

__main__.City

In [7]:
# Create an agent that must return `City`.
agent = Agent(MODEL_ID, output_type=City)
agent
# The agent is configured to validate model output against class `City`.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class '__main__.City'>, instrument=None)

In [8]:
# Run the minimal example agent.
result = agent.run_sync("Tell me about Paris")

result.output
# The result is a validated `City` object.

RuntimeError: This event loop is already running

# Resolving the Above RuntimeError in Jupyter

- Key thing to remember: Jupyter already runs an active event loop

- `agent.run_sync()` can raise a `RuntimeError` in notebook environments
- `nest_asyncio` patches the notebook event loop so nested async execution can work
- After `nest_asyncio.apply()`, async `PydanticAI` examples can run inside notebook cells

In [9]:
# Enable nested event loops for notebook execution.
nest_asyncio.apply()
nested_event_loop_enabled = True
_LOG.info("Nested event loop support enabled.")
nested_event_loop_enabled
# Async PydanticAI examples can now run from notebook cells.

Nested event loop support enabled.


True

- Re-run the previous cell that raised the `RuntimeError`

# Structured Outputs with Pydantic

- `PydanticAI` turns LLM responses into structured data
- Structured outputs help you:
  - Store validated outputs in databases
  - Feed typed objects into analytics
  - Pass structured data downstream without brittle string parsing

In [10]:
# Define a product schema for structured extraction.
class Product(BaseModel):
    name: str
    price: float
    category: str


Product
# The schema captures the product fields we want to extract.

__main__.Product

In [11]:
# Create an agent that must return `Product`.
agent = Agent(MODEL_ID, output_type=Product)
agent
# The agent is configured to return product data with typed fields.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class '__main__.Product'>, instrument=None)

In [12]:
# Ask the model for structured product information.
agent.run_sync("Describe the Apple AirPods Pro").output
# The response is validated as a `Product` class object.

Product(name='Apple AirPods Pro', price=249.0, category='Electronics/Audio')

# Validation and Retries

- Real LLM outputs are inconsistent
- Schema validation checks the generated structure
- Retries let `PydanticAI` ask the model to repair invalid output
- This notebook avoids custom parsing and retry logic in each prompt

In [13]:
# Define a schema that requires an integer age.
class Person(BaseModel):
    name: str
    age: int


Person
# The schema enforces integer typing for age values.

__main__.Person

In [14]:
# Configure retries so schema validation failures can be corrected.
agent = Agent(MODEL_ID, output_type=Person, retries=2)
agent
# The agent can retry when model output does not match `Person`.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class '__main__.Person'>, instrument=None)

In [15]:
# Run the retry-enabled agent.
agent.run_sync("Tell me about Albert Einstein")
# The result is a validated `Person` run result.

AgentRunResult(output=Person(name='Albert Einstein', age=76))

# Tools

- Agents can call Python functions as tools
- Tools let the model interact with real functions and external systems
- Tools are useful for APIs, databases, calculations, and deterministic helpers
- Tool calls reduce the chance that the model invents facts

In [16]:
# Create an agent with a deterministic weather tool.
agent = Agent(MODEL_ID, tools=[utils.get_weather])
agent
# The agent can call `utils.get_weather()` while answering.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class 'str'>, instrument=None)

In [17]:
# Ask a question that should use the weather tool.
agent.run_sync("What is the weather in Tokyo?")
# The run result includes the tool-backed weather answer.

AgentRunResult(output='The weather in Tokyo is sunny.')

# Dependencies

- Dependencies inject runtime context into agents and tools
- Example dependency values:
  - Tenant IDs
  - API clients
  - Feature flags
  - Environment context
- Dependencies let tools access context without global variables or prompt string formatting

In [18]:
# Define the dependency object passed into the agent at run time.
@dataclass
class Config:
    company: str


Config
# The dependency schema describes runtime context available to tools.

__main__.Config

In [19]:
# Create an agent that receives `Config` dependencies.
# `deps_type=Config` declares the shape of runtime context the agent can receive.
agent = Agent(MODEL_ID, deps_type=Config, tools=[utils.company_name])
agent
# Tools can access `Config` through the PydanticAI run context.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class 'str'>, instrument=None)

In [20]:
# Run the dependency-aware agent with a concrete configuration.
result = agent.run_sync(
    "What company is configured?", deps=Config(company="OpenAI")
)
result.output
# The answer reflects the runtime dependency value.

'The configured company is OpenAI.'

# Advanced Features

- The following sections demonstrate more advanced `PydanticAI` capabilities
- These features are useful for production-grade systems:
  - Custom validation
  - Streaming outputs
  - Model configuration
  - Usage tracking
  - Runtime limits
- Beginners can safely skip this section on a first read

# Result Validators

- Result validators are used to check model outputs after schema validation
- `Pydantic` validates structure automatically, but result validators enforce business rules
- A response can match the `Pydantic` schema and still fail logical constraints
- For example, this output may be valid according to the schema:
    - it has an `answer`
    - it has a `sources` list
- But it can still be logically wrong if:
    - the source list is empty
    - the `doc_id` does not exist
    - the quote does not actually appear in the cited document

- Result validators handle this second layer of validation

## Validation Flow

- Validation happens in two stages:
  - `Schema validation`: the model output must match `AnswerWithSources`
  - `Business-rule validation`: the registered `output_validator` enforces citation quality rules that schema alone cannot enforce
- Execution order:
  ```mermaid
  flowchart LR
      A[Model Output] --> B[Pydantic Schema Validation]
      B --> C[output_validator]
      C --> D[Final Result]
  ```

In [21]:
# Define source citation schemas with explicit references for validator examples.
class SourceRef(BaseModel):
    doc_id: str
    quote: str


class AnswerWithSources(BaseModel):
    answer: str
    sources: list[SourceRef]


AnswerWithSources
# The schemas describe answers that include source citations.

__main__.AnswerWithSources

## Prepare Validation Context

- We fetch the list of valid document IDs and include it in the agent instructions
- This helps:
    - reduce hallucinated references
    - constrain the model to known documents

In [22]:
# Build validator instructions from local document ids.
available_doc_ids = utils.get_available_document_ids()
# Build instructions that restrict citations to the local dataset.
validator_instructions = (
    "Use the search_documents tool to retrieve evidence from local documents. "
    f"Cite only these doc ids: {available_doc_ids}. "
    "For each source, copy the quote text exactly from tool output."
)
{
    "available_doc_ids": available_doc_ids,
    "validator_instruction_length": len(validator_instructions),
}
# The instructions constrain citations to the local document ids.

{'available_doc_ids': ['api',
  'billing',
  'integrations',
  'limits',
  'overview',
  'security',
  'support',
  'troubleshooting'],
 'validator_instruction_length': 260}

### Create the Validator Agent
- This agent:
    - generates structured output
    - retrieves documents using a tool
    - follows constrained citation rules



In [23]:
# Create an agent that returns answers with source references.
# The agent uses structured output plus the local document-search tool.
validator_agent = Agent(
    MODEL_ID,
    output_type=AnswerWithSources,
    instructions=validator_instructions,
    tools=[utils.search_documents],
)
validator_agent
# The validator agent can retrieve documents and return cited answers.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class '__main__.AnswerWithSources'>, instrument=None)

## Add Result Validator

- The `@output_validator` runs after schema validation and enforces business rules:
    - sources must be present
    - document IDs must exist
    - quotes must match source documents
    - duplicates are not allowed
- If validation fails, `ModelRetry` is raised, and the model is asked to generate a corrected answer.

In [24]:
# Register a result validator that checks citations against local documents.
@validator_agent.output_validator
def _validate_answer_sources(
    result: AnswerWithSources,
) -> AnswerWithSources:
    # Validate citations against the local document dataset.
    validated_result = utils.validate_document_sources(result)
    return validated_result


{"validator_registered": True}
# The validator agent now enforces schema and source-reference rules.

{'validator_registered': True}

## Manual Failure Example

- We intentionally create an invalid output to demonstrate how the validator triggers a retry.
- This example bypasses the model and directly tests the validator logic.

In [26]:
# Build an invalid answer object for the validator demo.
bad_answer = AnswerWithSources(
    answer="PydanticAI supports structured outputs.",
    sources=[],
)
bad_answer
# The invalid answer is missing source citations.

In [ ]:
# Trigger the validator on the intentionally invalid answer.
_LOG.info("Triggering the validator with an intentionally invalid answer.")
_validate_answer_sources(bad_answer)
# The validator raises `ModelRetry` for the missing sources.

## Run the Agent

- The agent will:
    - Generate structured output
    
    - Validate it against the schema
    
    - Apply business rules
    
    - Retry automatically if validation fails

In [27]:
# Run the validator agent with the local document search tool.
validator_result = asyncio.run(utils.run_validator_example(validator_agent))
validator_result
# The validator agent returns a cited answer that passed validation.

AnswerWithSources(answer='Atlas billing plans include Team and Enterprise plans, which offer features such as two-factor authentication (2FA). Billing details such as invoices can be managed and downloaded through the Settings > Billing section in the Atlas interface. Specific pricing or other plan tiers are not detailed in the provided documents. For exact billing options and plan details, accessing your Atlas settings or contacting support would be recommended.', sources=[SourceRef(doc_id='security', quote='Atlas supports two-factor authentication (2FA) for Team and Enterprise plans.'), SourceRef(doc_id='billing', quote='- You can download invoices from Settings > Billing.')])

# Streaming

- Streaming returns tokens as the model generates them
- Streaming benefits:
  - Lower perceived latency
  - Better user experience in chat interfaces
  - Progressive display of responses

In [39]:
# Create an agent for the streaming example.
stream_agent = Agent(
    MODEL_ID, instructions="Write one short paragraph about unit tests."
)
stream_agent
# The streaming agent is ready to produce incremental text.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings=None, output_type=<class 'str'>, instrument=None)

In [40]:
# Run the streaming helper and return the final result.
asyncio.run(utils.run_streaming_demo(stream_agent))
# The helper logs streamed text and returns the final result.

Streaming output:
Unit tests are automated tests that verify the functionality of individual components or units of code, such as functions or methods, in isolation from the rest of the application. Their primary purpose is to ensure that each unit performs as expected, helping developers catch bugs early, improve code quality, and simplify maintenance. By running unit tests frequently during development, teams can identify issues quickly and confidently make changes without introducing new errors.


'Unit tests are automated tests that verify the functionality of individual components or units of code, such as functions or methods, in isolation from the rest of the application. Their primary purpose is to ensure that each unit performs as expected, helping developers catch bugs early, improve code quality, and simplify maintenance. By running unit tests frequently during development, teams can identify issues quickly and confidently make changes without introducing new errors.'

# Provider Configuration

- Model objects let you configure providers directly, such as `base_url`
- Use an explicit model object when provider-specific options are needed


In [41]:
# Build an explicit provider model object when the installed API supports it.
explicit_model = utils.build_explicit_openai_model(MODEL_ID)
# Log which provider configuration path is active.
if explicit_model is None:
    _LOG.info("Explicit model unavailable; using string model ID.")
else:
    _LOG.info("Using explicit model object.")
{"explicit_model_available": explicit_model is not None}
# Provider configuration is either explicit or falls back to `MODEL_ID`.

Using OpenAI model with model_name='gpt-4.1-mini'.
Using explicit model object.


{'explicit_model_available': True}

In [42]:
# Run an agent with the explicit provider model when available.
agent = Agent(explicit_model or MODEL_ID, instructions="Be concise.")
result = asyncio.run(agent.run("Say hello in one sentence."))
result
# The result confirms that the provider configuration can execute a request.

AgentRunResult(output='Hello! How can I assist you today?')

# AgentRun

- `AgentRun` objects contain metadata about an agent execution
- `AgentRun` metadata includes:
  - Token usage
  - Message history
  - Tool calls
  - Final output
- Run metadata helps with:
  - Observability: inspect messages and tool calls
  - Cost tracking: inspect token usage
  - Governance: keep execution details available for review

In [43]:
# Run an agent and collect execution metadata.
meta_agent = Agent(MODEL_ID, instructions="Answer in one sentence.")
result = asyncio.run(meta_agent.run("What is a unit test?"))
# Extract execution metadata that helps inspect the run.
usage = getattr(result, "usage", None)
message_count = len(result.new_messages())
run_metadata = {
    "output": result.output,
    "messages_new": message_count,
    "usage": usage,
}
run_metadata
# The metadata summarizes output, message count, and usage details.

{'output': 'A unit test is a type of software test that verifies the correctness of a small, specific part of an application, typically a single function or method, to ensure it behaves as expected.',
 'messages_new': 2,
 'usage': <bound method AgentRunResult.usage of AgentRunResult(output='A unit test is a type of software test that verifies the correctness of a small, specific part of an application, typically a single function or method, to ensure it behaves as expected.')>}

# Usage Limits and Model Settings

- Usage limits help control:
  - API cost
  - Runaway loops
  - Excessive token usage
- `PydanticAI` supports safety and cost controls for production LLM systems

In [44]:
# Load version-tolerant classes for model settings and usage limits.
ModelSettings, UsageLimits = utils.get_settings_classes()
_LOG.info("Loaded ModelSettings and UsageLimits classes.")
{
    "model_settings_class": ModelSettings.__name__,
    "usage_limits_class": UsageLimits.__name__,
}
# The installed PydanticAI version determines where these classes come from.

Loaded ModelSettings and UsageLimits classes.


{'model_settings_class': 'ModelSettings', 'usage_limits_class': 'UsageLimits'}

In [45]:
# Create an agent with deterministic model settings.
settings_agent = Agent(
    MODEL_ID,
    instructions="Answer in a single sentence.",
    model_settings=ModelSettings(temperature=0.2),
)
settings_agent
# The agent has a low-temperature model setting.

Agent(model=OpenAIChatModel(), name=None, end_strategy='early', model_settings={'temperature': 0.2}, output_type=<class 'str'>, instrument=None)

In [46]:
# Run the settings example with a request limit.
result = asyncio.run(
    settings_agent.run(
        "Explain what unit tests are.",
        usage_limits=UsageLimits(request_limit=3),
    )
)

# Show the constrained response text.
result.output
# The response was generated with model settings and usage limits applied.

'Unit tests are automated tests that verify the correctness of individual components or functions of a software application in isolation.'

# Troubleshooting

- Missing API key: set `OPENAI_API_KEY` or the provider-specific key
- Event loop errors in notebooks: use `await agent.run(...)` instead of `run_sync`
- Validation errors: revise `output_type` or the validator to match expected output
